**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Performance Engineering & the Roofline Model

The unifying diagram behind every 'why is this slow' conversation in [GPU](../Intro_GPU/README.md), [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb), and [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb): measure your machine's compute roof and memory roof, place your kernels on the chart, and *know* which wall you're hitting before touching a line of code.

## 1. Pre-requisites

[Intro to GPU Systems](../Intro_GPU/Intro_GPU.ipynb) (the CGMA idea), [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) (caches exist).

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def bench(fn, *args, reps=7):
    ts = []
    for _ in range(reps):
        tic = time.perf_counter(); fn(*args); ts.append(time.perf_counter()-tic)
    return min(ts)                                  # min = least OS interference

---
### 🕐 Session 1 of 2 — *Measuring Your Machine's Roofs* (~40 min)
**Goal:** peak FLOP/s from matmul, peak GB/s from streaming — the two ceilings of all performance.
**Feeds into:** Session 2 (placing kernels on the roofline).

---

## 2. Two Ceilings

💡 **Intuition.** Every kernel is limited by one of two machine properties: how fast it can **compute** (FLOP/s, ceiling set by matmul-class code) or how fast it can **move data** (bytes/s, ceiling set by streaming). Which one binds is decided by the kernel's **arithmetic intensity** — FLOPs per byte touched — the same quantity [Intro_GPU](../Intro_GPU/Intro_GPU.ipynb) called CGMA. Below the machine's critical intensity, no cleverness in the arithmetic helps: you are paying for trucks, not workers.

In [ ]:
# roof 1: compute (large matmul → BLAS at near-peak)
# roof 2: memory bandwidth (pure streaming: y = x copy/scale of a cache-busting array)

# YOUR CODE HERE


---
### 🕐 Session 2 of 2 — *Kernels on the Roofline* (~40 min)
**Goal:** place real operations on the chart; watch intensity, not effort, decide their fate.
**Builds on:** Session 1.

---

## 3. The Chart That Ends Arguments

In [ ]:
# measure several kernels: achieved FLOP/s vs arithmetic intensity
# saxpy: 2 FLOPs per 12 bytes → intensity 0.167 (hopelessly memory-bound)
# elementwise exp: ~1 'FLOP' per 8 bytes (in truth many flops inside exp — we count 1 op)
# small matmul (fits cache) vs large: same math, different effective intensity

# YOUR CODE HERE


💡 **Intuition.** The diagnosis is now mechanical: a kernel far *below* its roof has implementation problems (fix the code); a kernel *on* a memory roof can only be helped by **raising its intensity** — fuse operations, tile for cache ([HW-Accelerated Computing's](../Intro_GPU/HW_Accelerated_Computing.ipynb) shared-memory story), or change algorithm. Optimizing a memory-bound kernel's arithmetic is polishing the truck's engine while it waits at the loading dock.

**The habit:** before optimizing anything, compute its intensity on a napkin and place it on this chart. Half of all optimization effort in the wild is spent on the wrong side of the critical intensity.

## 4. Conclusion

Two measured roofs, one intensity axis, every kernel placed — and the fix (better code vs more reuse vs different algorithm) read directly off the chart.

---
## Where next

- [HW-Accelerated Computing](../Intro_GPU/HW_Accelerated_Computing.ipynb) — raising intensity with shared memory.
- [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — rooflines at training scale.